# Dock the experimental shortlist with the validated conformer-ensemble protocol

Flexible Vina docking could not reproduce the lisinopril crystal pose, whereas the **conformer-ensemble protocol** (conformers generated from the SMILES, each docked rigidly, the best refined by flexible local minimisation) did in 2 of 3 redocking seeds (notebook 4). This notebook applies that protocol, unchanged, to the compounds in `experiment_plan.csv` (notebook 9) and to a lisinopril control.

**What this does and does not give you**
- The protocol is only **weakly validated**, and only for *pose reproduction of one ligand*. The score margins were small (the status text below reports them), and one of three seeds missed the crystal pose because no generated conformer was close enough.
- Pose reproduction does **not** show that the score ranks compounds by affinity; that is what notebook 7 tests. Until it reports `USEFUL_BEYOND_SIZE` for this protocol, treat the scores as reported-only information.
- Each compound is docked with several seeds (a different conformer set each) and the **best score over seeds** is reported, together with the per-seed values. In the lisinopril validation the best-of-three-seeds pose was the correct one.
- Cost: about `conformers x (rigid docking time)` per compound per seed, i.e. minutes per compound per seed at best, often much more. The notebook prints a running estimate and resumes from a checkpoint.

In [ ]:
# Colab-only dependency check. Nothing runs on VS Code, or when the tools are already present (see notebook 0 / requirements.txt).
import importlib.util, shutil, subprocess, sys
if importlib.util.find_spec('google.colab'):
    if shutil.which('mk_prepare_receptor.py') is None or importlib.util.find_spec('gemmi') is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'meeko==0.8.0', 'gemmi', 'rdkit', 'biopython'])  # ProDy is not needed (--read_pdb)
    if shutil.which('vina') is None:
        subprocess.check_call(['apt-get', 'install', '-y', '-qq', 'autodock-vina'])  # Colab (Ubuntu 24.04) ships Vina 1.2.5


In [2]:
from pathlib import Path
import os, sys, json, warnings
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    ROOT = Path('/content/drive/MyDrive/ACE_Workflow')
except ImportError:
    ROOT = Path(os.environ.get('ACE_WORKFLOW_ROOT', Path.cwd())).resolve()
if not (ROOT / 'ace_config.py').exists():
    ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'ace_config.py').exists()), ROOT)
if not (ROOT / 'ace_config.py').exists():
    raise FileNotFoundError('Could not find ace_config.py. Mount Drive or set ACE_WORKFLOW_ROOT.')
os.environ['ACE_WORKFLOW_ROOT'] = str(ROOT)
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
import ace_config

import hashlib, shutil, subprocess, time
import numpy as np, pandas as pd
from rdkit import Chem, RDLogger
RDLogger.DisableLog('rdApp.*')

# the docking code lives in a small importable module so notebooks 5b and 7 run exactly the same protocol
(ROOT / 'ace_ensemble_dock.py').write_text('"""Conformer-ensemble rigid docking followed by flexible local refinement (the protocol validated in notebook 4).\n\nKept as an importable module so notebooks 4-style redocking, candidate docking (5b) and the screening-power test (7)\nall run exactly the same code.\n"""\nimport re\nimport shutil\nimport subprocess\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom rdkit import Chem\nfrom rdkit.Chem import AllChem, rdDistGeom\nfrom scipy.spatial.distance import cdist\n\nPROTOCOL_NAME = \'conformer_ensemble_rigid_docking_then_flexible_refinement\'\n\n\ndef _lines(path, first_model_only=True):\n    out, started = [], False\n    for line in Path(path).read_text(errors=\'replace\').splitlines():\n        if line.startswith(\'MODEL\'):\n            started = True\n            continue\n        if started and line.startswith(\'ENDMDL\') and first_model_only:\n            break\n        if line.startswith((\'ATOM\', \'HETATM\')):\n            out.append(line)\n    return out\n\n\ndef _heavy(line):\n    return line[77:79].strip().upper() not in {\'H\', \'HD\', \'HS\'}\n\n\ndef _xyz(line):\n    return [float(line[30:38]), float(line[38:46]), float(line[46:54])]\n\n\ndef _to_template_order(input_lines, pose_lines, conf_heavy_xyz):\n    """Map PDBQT heavy atoms (Meeko order) onto the SMILES atom order via the INPUT conformer coordinates."""\n    inp = np.array([_xyz(l) for l in input_lines if _heavy(l)])\n    pose = np.array([_xyz(l) for l in pose_lines if _heavy(l)])\n    d = cdist(inp, conf_heavy_xyz)\n    j = d.argmin(axis=1)\n    if d.min(axis=1).max() > 0.05 or len(set(j)) != len(j):\n        raise RuntimeError(\'Atom mapping between Meeko PDBQT and RDKit conformer failed.\')\n    out = np.empty_like(pose)\n    out[j] = pose\n    return out\n\n\nclass Reference:\n    """Crystal reference for a symmetry-aware heavy-atom RMSD (no superposition)."""\n\n    def __init__(self, template_mol, crystal_mol):\n        self.ref_xyz = crystal_mol.GetConformer().GetPositions()\n        self.matches = template_mol.GetSubstructMatches(crystal_mol, uniquify=False, useChirality=False, maxMatches=20000)\n        if not self.matches:\n            raise RuntimeError(\'Could not map the SMILES-based molecule onto the crystal ligand.\')\n        probe = np.zeros_like(self.ref_xyz)\n        for i, j in enumerate(self.matches[0]):\n            probe[j] = self.ref_xyz[i]\n        assert self.rmsd(probe) < 1e-6, \'RMSD mapping self-check failed\'\n\n    def rmsd(self, pose_xyz_template_order):\n        return min(float(np.sqrt(np.mean(np.sum((pose_xyz_template_order[list(m)] - self.ref_xyz) ** 2, axis=1)))) for m in self.matches)\n\n\nclass EnsembleDocker:\n    def __init__(self, receptor_pdbqt, box, vina, meeko_ligand, workdir, cpu=2):\n        self.receptor, self.box, self.vina, self.meeko = str(receptor_pdbqt), box, vina, meeko_ligand\n        self.workdir, self.cpu = Path(workdir), cpu\n        self.workdir.mkdir(parents=True, exist_ok=True)\n\n    def _box_args(self, pad=0.0):\n        b = self.box\n        return [\'--center_x\', str(b[\'center_x\']), \'--center_y\', str(b[\'center_y\']), \'--center_z\', str(b[\'center_z\']),\n                \'--size_x\', str(b[\'size_x\'] + pad), \'--size_y\', str(b[\'size_y\'] + pad), \'--size_z\', str(b[\'size_z\'] + pad)]\n\n    def _vina(self, ligand, extra, pad=0.0):\n        r = subprocess.run([self.vina, \'--receptor\', self.receptor, \'--ligand\', str(ligand), *self._box_args(pad), \'--cpu\', str(self.cpu), *extra],\n                           text=True, capture_output=True)\n        if r.returncode:\n            raise RuntimeError((r.stderr or r.stdout)[-800:])\n        return r.stdout\n\n    def dock(self, smiles, tag, seed, n_conformers=100, rigid_exhaustiveness=4, refine_top_k=20, reference=None, keep_intermediate=False):\n        """Return a dict with the best refined score/pose; a \'refined\' DataFrame lists the refined conformers.\n\n        Undefined stereocentres are embedded in both configurations (the ensemble contains both), so the reported score is the\n        best over the possible stereoisomers.\n        """\n        template = Chem.MolFromSmiles(smiles)\n        if template is None:\n            return {\'status\': \'INVALID_SMILES\', \'tag\': tag, \'seed\': seed}\n        mol_h = Chem.AddHs(template)\n        params = rdDistGeom.ETKDGv3()\n        params.randomSeed = int(seed)\n        params.pruneRmsThresh = 0.5\n        params.numThreads = self.cpu\n        cids = list(rdDistGeom.EmbedMultipleConfs(mol_h, n_conformers, params))\n        if not cids:\n            params.useRandomCoords = True\n            cids = list(rdDistGeom.EmbedMultipleConfs(mol_h, n_conformers, params))\n        if not cids:\n            return {\'status\': \'FAILED_EMBED\', \'tag\': tag, \'seed\': seed}\n        try:\n            AllChem.MMFFOptimizeMoleculeConfs(mol_h, maxIters=1000, numThreads=self.cpu)\n        except Exception:\n            pass\n        n_heavy = template.GetNumHeavyAtoms()\n        d = self.workdir / f\'{tag}_s{seed}\'\n        d.mkdir(parents=True, exist_ok=True)\n\n        records, store = [], {}\n        for cid in cids:\n            name = f\'c{cid:04d}\'\n            sdf, flex = d / f\'{name}.sdf\', d / f\'{name}.pdbqt\'\n            sdf.write_text(Chem.MolToMolBlock(mol_h, confId=cid))\n            r = subprocess.run([self.meeko, \'-i\', str(sdf), \'-o\', str(flex)], text=True, capture_output=True)\n            if r.returncode or not flex.exists():\n                continue\n            flex_text = flex.read_text().splitlines()\n            flex_atoms = _lines(flex, first_model_only=False)\n            rigid, out = d / f\'{name}_rigid.pdbqt\', d / f\'{name}_rigid_docked.pdbqt\'\n            rigid.write_text(\'\\n\'.join([l for l in flex_text if l.startswith(\'REMARK\')] + [\'ROOT\', *flex_atoms, \'ENDROOT\', \'TORSDOF 0\']) + \'\\n\')\n            try:\n                self._vina(rigid, [\'--exhaustiveness\', str(rigid_exhaustiveness), \'--num_modes\', \'3\', \'--seed\', str(seed), \'--out\', str(out)])\n            except RuntimeError:\n                continue\n            pose_lines = _lines(out)\n            score = next((float(x.split()[3]) for x in out.read_text().splitlines() if x.startswith(\'REMARK VINA RESULT\')), np.nan)\n            conf_heavy = mol_h.GetConformer(cid).GetPositions()[:n_heavy]\n            rec = {\'conformer\': cid, \'rigid_score\': score}\n            if reference is not None:\n                rec[\'rigid_rmsd_A\'] = reference.rmsd(_to_template_order(flex_atoms, pose_lines, conf_heavy))\n            records.append(rec)\n            store[cid] = {\'flex\': flex, \'pose\': pose_lines, \'atoms\': flex_atoms, \'conf_heavy\': conf_heavy}\n        if not records:\n            shutil.rmtree(d, ignore_errors=True)\n            return {\'status\': \'FAILED_DOCKING\', \'tag\': tag, \'seed\': seed}\n        ens = pd.DataFrame(records).sort_values(\'rigid_score\').reset_index(drop=True)\n\n        refined_rows = []\n        for row in ens.head(refine_top_k).itertuples():\n            o = store[row.conformer]\n            it, start_lines = iter(o[\'pose\']), []\n            for l in Path(o[\'flex\']).read_text().splitlines():\n                if l.startswith((\'ATOM\', \'HETATM\')):\n                    p = next(it)\n                    l = l[:30] + p[30:54] + l[54:]          # same atoms, rigid-pose coordinates\n                start_lines.append(l)\n            start = d / f\'c{row.conformer:04d}_start.pdbqt\'\n            start.write_text(\'\\n\'.join(start_lines) + \'\\n\')\n            refined = d / f\'c{row.conformer:04d}_refined.pdbqt\'\n            try:\n                txt = self._vina(start, [\'--local_only\', \'--out\', str(refined)], pad=12.0)   # larger box: edge poses may poke out of the docking box\n            except RuntimeError:\n                continue\n            rr = {\'conformer\': row.conformer, \'rigid_score\': row.rigid_score, \'refined_score\': float(re.findall(r\'Estimated Free Energy of Binding\\s*:\\s*(-?\\d+\\.?\\d*)\', txt)[-1]),\n                  \'refined_pdbqt\': str(refined)}\n            if reference is not None:\n                rr[\'refined_rmsd_A\'] = reference.rmsd(_to_template_order(o[\'atoms\'], _lines(refined), o[\'conf_heavy\']))\n            refined_rows.append(rr)\n        if not refined_rows:\n            shutil.rmtree(d, ignore_errors=True)\n            return {\'status\': \'FAILED_REFINEMENT\', \'tag\': tag, \'seed\': seed}\n        refined_df = pd.DataFrame(refined_rows).sort_values(\'refined_score\').reset_index(drop=True)\n        best = refined_df.iloc[0]\n        best_pose = self.workdir / f\'{tag}_s{seed}_best.pdbqt\'\n        shutil.copy2(best.refined_pdbqt, best_pose)\n        result = {\'status\': \'DONE\', \'tag\': tag, \'seed\': seed, \'n_conformers\': len(cids), \'n_rigid_docked\': len(ens), \'best_rigid_score\': float(ens.rigid_score.min()),\n                  \'best_refined_score\': float(best.refined_score), \'best_conformer\': int(best.conformer), \'best_pose_pdbqt\': str(best_pose),\n                  \'refined\': refined_df.drop(columns=\'refined_pdbqt\'), \'all_conformers\': ens}\n        if reference is not None:\n            result[\'top_rmsd_A\'] = float(best.refined_rmsd_A)\n            result[\'best_rigid_rmsd_A\'] = float(ens.rigid_rmsd_A.min())\n        if not keep_intermediate:\n            shutil.rmtree(d, ignore_errors=True)\n        return result\n', encoding='utf-8')
import importlib, ace_ensemble_dock as ed; importlib.reload(ed)

PATHS = ace_config.PATHS; RANDOM_STATE = ace_config.RANDOM_STATE; DOCKING_CPU = ace_config.CPU_WORKERS
VINA = shutil.which('vina') or shutil.which('vina.exe'); MEEKO_LIGAND = shutil.which('mk_prepare_ligand.py')
if not VINA or not MEEKO_LIGAND: raise RuntimeError('AutoDock Vina (>= 1.2) and Meeko must be on PATH.')
ANALYSIS = PATHS['REDOCKING_ANALYSIS']; REPORTS = PATHS['DOCKING_REPORTS']; REPORTS.mkdir(parents=True, exist_ok=True)
receptor_pdbqt = PATHS['REDOCKING_RECEPTOR'] / '1o86_receptor.pdbqt'
box = json.loads((ANALYSIS / '1o86_docking_box.json').read_text())

# --- the validated protocol (written by notebook 4 only if the redocking gate passed) ---
settings_file = ANALYSIS / '1o86_redocking_validated_settings.json'
if not settings_file.exists(): raise FileNotFoundError('No validated settings found. Run notebook 4 (ensemble redocking) until it passes, or this protocol is not validated.')
settings = json.loads(settings_file.read_text())
if settings.get('protocol') != ed.PROTOCOL_NAME: raise RuntimeError(f"Validated settings are for another protocol: {settings.get('protocol')}")
PROTOCOL = dict(n_conformers=settings['conformers_per_seed'], rigid_exhaustiveness=settings['rigid_exhaustiveness'], refine_top_k=settings['refine_top_k'])
SEEDS = settings['seeds']          # all validation seeds (including the one that failed), not only the passing ones
summary_file = ANALYSIS / '1o86_redocking_ensemble_summary.csv'
if summary_file.exists():
    sm = pd.read_csv(summary_file); n_pass, n_seeds = int(sm.top_pass_2A.sum()), len(sm); margin = float(sm.margin_vs_best_wrong_kcal.median())
    VALIDATION_STATUS = (f'WEAKLY VALIDATED for pose reproduction of lisinopril: top-ranked pose within 2 A in {n_pass}/{n_seeds} redocking seeds, median score margin '
                         f'{margin:.2f} kcal/mol; affinity ranking not validated here')
else:
    VALIDATION_STATUS = 'validated settings found, details unavailable'
print('Protocol:', PROTOCOL, '| seeds', SEEDS); print('Status :', VALIDATION_STATUS)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Protocol: {'n_conformers': 100, 'rigid_exhaustiveness': 4, 'refine_top_k': 20} | seeds [7, 42, 2026]
Status : WEAKLY VALIDATED for pose reproduction of lisinopril: top-ranked pose within 2 A in 2/3 redocking seeds, median score margin 0.35 kcal/mol; affinity ranking not validated here


In [3]:
# 1. Which compounds? Default: everything in the experiment plan (candidates + controls) from notebook 9
SELECTION = 'experiment_plan'     # or 'top_n' (top tier A/B chemicals from the evidence table)
TOP_N = 20
MAX_COMPOUNDS = None              # set a small number for a quick look
LISINOPRIL = 'NCCCC[C@H](N[C@@H](CCc1ccccc1)C(=O)O)C(=O)N1CCC[C@H]1C(=O)O'

def canon(s): return Chem.MolToSmiles(Chem.MolFromSmiles(s)) if Chem.MolFromSmiles(s) is not None else None
if SELECTION == 'experiment_plan':
    plan = pd.read_csv(PATHS['VIS_REPORTS'] / 'experiment_plan.csv'); comp = plan[['role', 'canonical_smiles']].copy()
else:
    ev = pd.read_csv(PATHS['VIS_REPORTS'] / 'candidate_evidence_table.csv')
    comp = ev[ev.evidence_tier.isin(['A', 'B'])].sort_values('p_active_raw', ascending=False).head(TOP_N)[['canonical_smiles']].assign(role='candidate')
comp['canonical_smiles'] = comp.canonical_smiles.map(canon); comp = comp.dropna(subset=['canonical_smiles'])
lis = canon(LISINOPRIL)
if lis not in set(comp.canonical_smiles): comp = pd.concat([pd.DataFrame([{'role': 'positive control (known ACE inhibitor)', 'canonical_smiles': lis}]), comp], ignore_index=True)
comp = comp.drop_duplicates('canonical_smiles').reset_index(drop=True)
comp = pd.concat([comp[comp.canonical_smiles == lis], comp[comp.canonical_smiles != lis]], ignore_index=True)   # lisinopril first: the quality check appears early
if MAX_COMPOUNDS: comp = comp.head(MAX_COMPOUNDS)
comp['compound_id'] = [('ENS_LISINOPRIL' if s == lis else f"ENS_{i:03d}_{hashlib.sha1(s.encode()).hexdigest()[:6]}") for i, s in enumerate(comp.canonical_smiles)]
comp['is_lisinopril'] = comp.canonical_smiles == lis
print(f'{len(comp)} compounds x {len(SEEDS)} seeds = {len(comp) * len(SEEDS)} ensemble dockings'); display(comp.role.value_counts())


6 compounds x 3 seeds = 18 ensemble dockings


,count
role,
"negative control (ChEMBL-reported inactive, biochemical)",3
positive control (known ACE inhibitor),2
candidate (tier A),1


In [4]:
# 2. Docking (resumable). The lisinopril control is compared with the crystal pose on every seed.
docker = ed.EnsembleDocker(receptor_pdbqt, box, VINA, MEEKO_LIGAND, PATHS['DOCKING_RUNS'] / 'ensemble', cpu=DOCKING_CPU)
crystal = Chem.RemoveHs(Chem.SDMolSupplier(str(PATHS['REDOCKING_LIGAND'] / '1o86_LPR_crystal.sdf'), removeHs=False)[0])
reference = ed.Reference(Chem.MolFromSmiles(lis), crystal)

ckpt_file = PATHS['DOCKING_RUNS'] / 'ensemble_checkpoint.csv'
records = pd.read_csv(ckpt_file).query("status == 'DONE'").to_dict('records') if ckpt_file.exists() else []
done = {(r['compound_id'], int(r['seed'])) for r in records}
n_total, n_done0, t0 = len(comp) * len(SEEDS), len(records), time.time()
print(f'{n_total} runs ({n_done0} already done).')
for row in comp.itertuples():
    for seed in SEEDS:
        if (row.compound_id, seed) in done: continue
        t = time.time()
        r = docker.dock(row.canonical_smiles, row.compound_id, seed, **PROTOCOL, reference=reference if row.is_lisinopril else None)
        rec = {'compound_id': row.compound_id, 'seed': seed, 'status': r['status'], 'runtime_s': round(time.time() - t), 'n_conformers': r.get('n_conformers', np.nan),
               'best_rigid_score': r.get('best_rigid_score', np.nan), 'best_refined_score': r.get('best_refined_score', np.nan), 'best_pose_pdbqt': r.get('best_pose_pdbqt', ''),
               'top_rmsd_A_vs_crystal': r.get('top_rmsd_A', np.nan), 'best_rigid_rmsd_A_vs_crystal': r.get('best_rigid_rmsd_A', np.nan)}
        records.append(rec); pd.DataFrame(records).to_csv(ckpt_file, index=False)
        n_new = len(records) - n_done0; per = (time.time() - t0) / n_new
        print(f"{row.compound_id} seed {seed}: {r['status']}" + (f", score {r['best_refined_score']:.2f}" if r['status'] == 'DONE' else '') + (f", RMSD to crystal {r['top_rmsd_A']:.2f} A" if 'top_rmsd_A' in r else '') +
              f"  [{len(records)}/{n_total}; {per / 60:.1f} min/run; ~{(n_total - len(records)) * per / 3600:.1f} h left]")
res = pd.DataFrame(records)
print(res.status.value_counts().to_dict())


18 runs (18 already done).
{'DONE': 18}


In [5]:
# 3. Aggregate: best score over seeds (per-seed values kept), control check, yardstick, validation stamp
ok = res[res.status == 'DONE'].copy()
best_idx = ok.groupby('compound_id').best_refined_score.idxmin()
best = ok.loc[best_idx].set_index('compound_id')
g = ok.groupby('compound_id').best_refined_score
agg = pd.DataFrame({'dock_score_kcal': best.best_refined_score, 'dock_best_seed': best.seed, 'dock_best_pose_pdbqt': best.best_pose_pdbqt,
                    'dock_mean_score_kcal': g.mean(), 'dock_sd_score_kcal': g.std(ddof=1), 'dock_n_seeds': g.size()}).reset_index()
per_seed = ok.pivot(index='compound_id', columns='seed', values='best_refined_score'); per_seed.columns = [f'score_seed_{c}' for c in per_seed.columns]
out = comp.merge(agg, on='compound_id', how='left').merge(per_seed.reset_index(), on='compound_id', how='left')
Path(PATHS['DOCKING_POSES']).mkdir(parents=True, exist_ok=True)
for r in out.dropna(subset=['dock_best_pose_pdbqt']).itertuples():
    shutil.copy2(r.dock_best_pose_pdbqt, Path(PATHS['DOCKING_POSES']) / f'{r.compound_id}_ensemble_best.pdbqt')

ctrl = res[(res.compound_id == 'ENS_LISINOPRIL') & (res.status == 'DONE')]
ctrl_summary = {}
if len(ctrl):
    bs = ctrl.loc[ctrl.best_refined_score.idxmin()]
    ctrl_summary = {'per_seed_top_rmsd_A': {int(r.seed): round(float(r.top_rmsd_A_vs_crystal), 2) for r in ctrl.itertuples()}, 'best_of_seeds_seed': int(bs.seed), 'best_of_seeds_rmsd_A': round(float(bs.top_rmsd_A_vs_crystal), 2)}
    print('Lisinopril control vs crystal pose:', ctrl_summary)
    print('  -> best-of-seeds pose ' + ('REPRODUCES' if bs.top_rmsd_A_vs_crystal <= 2.0 else 'does NOT reproduce') + ' the crystal pose in this run.')
ref_score = float(out.loc[out.is_lisinopril, 'dock_score_kcal'].iloc[0]) if out.loc[out.is_lisinopril, 'dock_score_kcal'].notna().any() else np.nan
DG_TOLERANCE_KCAL = 1.0
out['dg_vs_lisinopril_kcal'] = out.dock_score_kcal - ref_score
out['dg_acceptable'] = out.dock_score_kcal <= ref_score + DG_TOLERANCE_KCAL        # yardstick only (lisinopril + tolerance, fixed in advance)
out['docking_protocol'] = ed.PROTOCOL_NAME; out['docking_validation'] = VALIDATION_STATUS
out.to_csv(REPORTS / 'ensemble_docking_results.csv', index=False)
(REPORTS / 'ensemble_docking_metadata.json').write_text(json.dumps({'protocol': ed.PROTOCOL_NAME, 'settings': settings, 'seeds': SEEDS, 'validation_status': VALIDATION_STATUS, 'lisinopril_control': ctrl_summary,
    'lisinopril_score_kcal': ref_score, 'dg_tolerance_kcal': DG_TOLERANCE_KCAL, 'receptor_sha256': ace_config.sha256_file(receptor_pdbqt)}, indent=2, default=float))
(REPORTS / 'ensemble_docking_disclaimer.txt').write_text(
    'Docking used a conformer-ensemble rigid-docking protocol with flexible refinement. ' + VALIDATION_STATUS + '. '
    'Scores are not validated predictors of binding affinity; whether they separate ACE actives from inactives is tested separately (notebook 7). They cannot by themselves confirm that a compound inhibits ACE; experimental assays are required.\n')
print(VALIDATION_STATUS)
show = ['compound_id', 'role', 'dock_score_kcal', 'dock_mean_score_kcal', 'dock_n_seeds', 'dg_vs_lisinopril_kcal', 'dg_acceptable']
display(out[show].sort_values('dock_score_kcal').round(2))


Lisinopril control vs crystal pose: {'per_seed_top_rmsd_A': {7: 1.3, 42: 8.8, 2026: 0.93}, 'best_of_seeds_seed': 2026, 'best_of_seeds_rmsd_A': 0.93}
  -> best-of-seeds pose REPRODUCES the crystal pose in this run.
WEAKLY VALIDATED for pose reproduction of lisinopril: top-ranked pose within 2 A in 2/3 redocking seeds, median score margin 0.35 kcal/mol; affinity ranking not validated here


,compound_id,role,dock_score_kcal,dock_mean_score_kcal,dock_n_seeds,dg_vs_lisinopril_kcal,dg_acceptable
4,ENS_004_ecf793,"negative control (ChEMBL-reported inactive, bi...",-8.53,-8.48,3,-1.08,True
1,ENS_001_304210,candidate (tier A),-7.72,-7.44,3,-0.26,True
5,ENS_005_dc66b4,"negative control (ChEMBL-reported inactive, bi...",-7.59,-7.37,3,-0.13,True
0,ENS_LISINOPRIL,positive control (known ACE inhibitor),-7.46,-7.03,3,0.00,True
3,ENS_003_52bcb1,"negative control (ChEMBL-reported inactive, bi...",-7.21,-7.20,3,0.25,True
2,ENS_002_d79d53,positive control (known ACE inhibitor),-5.45,-5.44,3,2.00,False
